## Tool definition

In [3]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [4]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [5]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [6]:
tool1.invoke({"x": 467})

21.61018278497431

## Adding to agents

In [7]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(model_provider="ollama", model="qwen3:8b")

agent = create_agent(
    model=model,
    tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [8]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

<think>
Okay, the user asked for the square root of 467. I used the square_root function with x=467 and got the result approximately 21.61018278497431. Now I need to present this answer clearly. Let me check if the user might need the square as well, but they only asked for the square root. Wait, maybe they want both? Let me see the original question again. No, the user specifically asked for the square root. So I'll provide that value, rounded to a reasonable decimal place. Let me confirm the calculation to ensure accuracy. Yes, 21.61 squared is about 467. So the answer is correct. I'll format the response to show the square root of 467 is approximately 21.61.
</think>

The square root of 467 is approximately **21.61**. 

If you'd also like the square of 467, it is **217,489**. Let me know!


In [9]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='eb6cd6f7-9918-48cf-b85f-67a0c78d082f'),
 AIMessage(content='<think>\nOkay, the user is asking for the square root of 467. Let me check if I have the right tool for that. The available function is square_root, which takes a number x. So I need to call that function with x set to 467. I should make sure that 467 is a valid number, which it is. No other parameters are needed. Just pass 467 into the square_root function and return the result.\n</think>\n\n', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-01-20T19:01:23.2104697Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4566735300, 'load_duration': 2052132500, 'prompt_eval_count': 169, 'prompt_eval_duration': 194661300, 'eval_count': 123, 'eval_duration': 2311525100, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--b04718fb-60de-46f2-b955-63c9f85f696b-0

In [10]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'f2e371a3-6f95-4f6d-99b8-d8a8e4db6dc3', 'type': 'tool_call'}]


## Without web search

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model=model
)

In [12]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [13]:
print(response['messages'][-1].content)

<think>
Okay, the user is asking how up to date my training knowledge is. I need to explain that my training data is up to 2024, but I should also mention that my knowledge might not be current beyond that. I should clarify that while my training data is recent, there could be new information or developments that I'm not aware of. It's important to highlight that I can still provide helpful information based on what I know, but I should advise the user to verify critical information with reliable sources. I should make sure to keep the response clear and straightforward, avoiding any technical jargon. Also, I need to check if there's any specific context or field the user is interested in, but since the question is general, I'll keep it broad. Let me structure the response to first state the cutoff date, then note the limitations, and finally offer assistance based on my training data.
</think>

My training knowledge is based on data up to 2024, but I should note that my understanding 

## Add web search tool

In [15]:
from dotenv import load_dotenv

load_dotenv()

True

In [16]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_San_Francisco',
   'title': 'Mayor of San Francisco - Wikipedia',
   'content': 'The current mayor is Democrat Daniel Lurie.',
   'score': 0.95324475,
   'raw_content': None},
  {'url': 'https://apnews.com/article/san-francisco-new-mayor-liberal-city-81ea0a7b37af6cbb68aea7ef5cc6a4f0',
   'title': "San Francisco's new mayor is starting to unite the fractured city",
   'content': 'San Francisco Mayor Daniel Lurie, a political newcomer and Levi Strauss heir, has marked his first 100 days with a hands-on, business-friendly approach.',
   'score': 0.8755427,
   'raw_content': None},
  {'url': 'https://www.sf.gov/departments--office-mayor',
   'title': 'Office of the Mayor - SF.gov',
   'content': 'Daniel Lurie is the 46th Mayor of the City and County of San Francisco.',
   'score': 0.8488849,
   'raw_content': None

In [17]:
agent = create_agent(
    model=model,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [18]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='998de26a-0edf-4a29-9f8b-d9f0d588b58e'),
 AIMessage(content='<think>\nOkay, the user is asking for the current mayor of San Francisco. I need to find the most up-to-date information. Since I don\'t have real-time data, I should perform a web search. Let me check the latest news or official government sources. The function provided is web_search, so I\'ll use that. The query should be straightforward: "current mayor of San Francisco". I\'ll make sure to use the function correctly and retrieve the latest information.\n</think>\n\n', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-01-20T22:04:15.7242673Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10447689400, 'load_duration': 7831658800, 'prompt_eval_count': 145, 'prompt_eval_duration': 232585500, 'eval_count': 121, 'eval_duration': 2382155200, 'logprobs': None, 'model_name': 'q